In [ ]:
import json
import requests
import pandas as pd
import re
import unicodedata
from datetime import datetime, timezone
from collections import defaultdict

# ============================================================================
# FUNÇÕES DE EXTRAÇÃO (API)
# ============================================================================


def hoje_utc_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")

def make_headers(token: str) -> dict:
    return {
        "Accept": "application/json, text/plain, */*",
        "Authorization": f"Bearer {token}",
        "User-Agent": "Mozilla/5.0 ...",
        "Content-Type": "application/json",
    }

def import_json_payload(path_json: str) -> str:
    with open(path_json, "r", encoding="utf-8") as f:
        json_dict = json.load(f)
    return json.dumps(json_dict, ensure_ascii=False)

def get_codCatalogo_from_payload(path_json: str) -> list[int]:
    with open(path_json, encoding="utf-8") as f:
        json_data = json.load(f)

    solicitacoes_nested = json_data.get("solicitacoes", [])

    solicitacoes_flat = []
    for item in solicitacoes_nested:
        if isinstance(item, list):
            solicitacoes_flat.extend(item)
        else:
            solicitacoes_flat.append(item)

    df = pd.DataFrame(solicitacoes_flat)
    if "codCatalogo" not in df.columns:
        return []
    # remove nulos e dupes preservando ordem
    cods = [int(x) for x in df["codCatalogo"].dropna().tolist()]
    seen = set()
    cods_unique = []
    for c in cods:
        if c not in seen:
            cods_unique.append(c)
            seen.add(c)
    return cods_unique

def fetch_dados_etapa(codigos: list[int], token: str) -> pd.DataFrame:
    url = "https://actogestaoapi-gdhrfgdfc8bbe8hs.brazilsouth-01.azurewebsites.net/api/RelatoriosEtapa/ObterTempoEtapaRelatorio"
    payload = {
        "codCatalogos": codigos,
        "dataInicio": "2022-01-01T00:00:00.000Z",
        "dataFim": hoje_utc_iso(),
        "ativo": 1,
    }
    r = requests.post(url, json=payload, headers=make_headers(token), timeout=60)
    r.raise_for_status()
    data = r.json()
    return pd.DataFrame(data if isinstance(data, list) else data.get("data", []))

def fetch_dados_solicitacoes(payload_str: str, token: str) -> pd.DataFrame:
    url = "https://actogestaoapi-gdhrfgdfc8bbe8hs.brazilsouth-01.azurewebsites.net/api/Tabela/VisualizarDadosIntermediarios"
    config = json.loads(payload_str)

    resp = requests.post(url, headers=make_headers(token), json=config, timeout=60)
    resp.raise_for_status()
    data = resp.json()

    lista_final = []
    if isinstance(data.get("data"), list):
        for item in data["data"]:
            dados_dict = item.get("dados", {})
            if isinstance(dados_dict, dict):
                for _, lista in dados_dict.items():
                    if isinstance(lista, list):
                        lista_final.extend(lista)

    return pd.DataFrame(lista_final)

def extrair_tabela_acto_gestao(path_payload_json: str, token: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    try:
        lista_cod_catalogo = get_codCatalogo_from_payload(path_payload_json)
        
        # Tenta buscar os dados, se a API falhar (Erro 500), assume DataFrame vazio
        try:
            df_etapas = fetch_dados_etapa(lista_cod_catalogo, token) if lista_cod_catalogo else pd.DataFrame()
        except Exception as e:
            print(f"AVISO: Falha ao buscar ETAPAS para {path_payload_json}: {e}")
            df_etapas = pd.DataFrame()

        try:
            df_solicitacoes = fetch_dados_solicitacoes(import_json_payload(path_payload_json), token)
        except Exception as e:
            print(f"AVISO: Falha ao buscar SOLICITAÇÕES para {path_payload_json}: {e}")
            df_solicitacoes = pd.DataFrame()

        print(f"Sucesso: {path_payload_json} | Solicitacoes: {len(df_solicitacoes)} | Etapas: {len(df_etapas)}")
        return df_solicitacoes, df_etapas
        
    except Exception as e:
        print(f"ERRO CRÍTICO no processamento de {path_payload_json}: {e}")
        return pd.DataFrame(), pd.DataFrame()


In [ ]:
# ============================================================================
# FUNÇÕES DE TRANSFORMAÇÃO (Limpeza de dados)
# ============================================================================

def tratar_nome_colunas(df: pd.DataFrame) -> pd.DataFrame:
    """Remove pipe com número, parênteses com número e dois pontos dos nomes de colunas."""
    df = df.copy()
    renomear = {}
    for c in df.columns:
        nome = str(c).strip()
        m = re.match(r"^(.+)\|(\d+)$", nome)
        if m:
            nome = m.group(1).strip()
        m2 = re.match(r"^(.+)\((\d+)\)$", nome)
        if m2:
            nome = m2.group(1).strip()
        nome = nome.rstrip(":").strip()
        renomear[c] = nome
    return df.rename(columns=renomear)


def colunas_para_snake_case(df: pd.DataFrame) -> pd.DataFrame:
    """Converte nomes das colunas para snake_case sem acentos."""
    def to_snake(s):
        s = str(s).strip()
        s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
        s = re.sub(r"[^a-zA-Z0-9\s]", " ", s)
        s = re.sub(r"\s+", "_", s).strip("_").lower()
        return s or "unnamed"
    return df.rename(columns={c: to_snake(c) for c in df.columns})


def converter_colunas_data(df: pd.DataFrame, colunas: list | None = None) -> pd.DataFrame:
    """Converte colunas especificadas para datetime (erros viram NaT)."""
    df = df.copy()
    if colunas is None:
        colunas = [c for c in df.columns if 'data' in c.lower()]
    for c in colunas:
        if c not in df.columns:
            continue
        df[c] = pd.to_datetime(df[c], errors="coerce")
    return df


def dropar_colunas_esparsas(df: pd.DataFrame, limite_nan_pct: float = 0.95) -> pd.DataFrame:
    """Remove colunas com percentual de NaNs >= limite_nan_pct."""
    df = df.copy()
    n = len(df)
    dropar = [c for c in df.columns if df[c].isna().sum() / n >= limite_nan_pct]
    return df.drop(columns=dropar)


def consolidar_conceito_bfill(df: pd.DataFrame) -> pd.DataFrame:
    """Consolida colunas com nomes duplicados usando bfill(axis=1)."""
    grupos = defaultdict(list)
    for i in range(len(df.columns)):
        grupos[df.columns[i]].append(i)
    first_idx = {}
    for i in range(len(df.columns)):
        nome = df.columns[i]
        if nome not in first_idx:
            first_idx[nome] = i
    result = {}
    for nome in sorted(first_idx.keys(), key=lambda n: first_idx[n]):
        indices = grupos[nome]
        if len(indices) == 1:
            result[nome] = df.iloc[:, indices[0]]
        else:
            sub = df.iloc[:, indices].bfill(axis=1)
            result[nome] = sub.iloc[:, 0]
    return pd.DataFrame(result, index=df.index)


print("✓ Funções carregadas: Extração (API) + Transformação (Limpeza)")